# DeepDTA Reproduction with Attention Mechanism
### Drug-Target Binding Affinity Prediction — Davis & KIBA Datasets

**Reference paper:** Öztürk, Özgür, Ozkirimli (2018). *DeepDTA: deep drug-target binding affinity prediction.* Bioinformatics, 34(17), i821-i829. [arXiv:1801.10193](https://arxiv.org/abs/1801.10193)

**Original repository:** https://github.com/hkmztrk/DeepDTA

---

**What this notebook does:**
1. Reproduces the original DeepDTA two-branch CNN architecture (SMILES branch + protein sequence branch)
2. Adds a custom **attention pooling layer** on top of each CNN branch (an extension beyond the original paper) so the model learns *which positions* in the drug/protein sequence matter most for a given prediction, instead of only taking the single strongest signal via max pooling
3. Trains and evaluates on both **Davis** (kinase inhibitors) and **KIBA** (larger, sparser benchmark) datasets
4. Reports the paper's own evaluation metrics: **MSE**, **Concordance Index (CI)**, and **rm²**
5. Visualizes attention weights to sanity-check what the model is focusing on

**Author's background note:** built as part of a deep learning portfolio roadmap (DeepDTA → GraphDTA → ProteinMPNN → ESM-2 → EquiBind → TankBind → DiffDock → RFdiffusion → AlphaFold3), following prior hands-on work in molecular docking (HADDOCK, HDOCK, AutoDock Vina) and MD simulation (GROMACS, OpenMM, AMBER).


## 1. Environment Setup

In [ ]:
!pip install -q tensorflow scikit-learn numpy pandas scipy matplotlib seaborn

In [ ]:
import tensorflow as tf
import numpy as np
import json, pickle, itertools
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Get the Data

The original DeepDTA repository ships both the Davis and KIBA datasets in the exact format used in the paper (SMILES dictionary, protein sequence dictionary, affinity matrix, and the official train/test fold splits). We clone it directly rather than sourcing the data separately, to guarantee an apples-to-apples comparison with the published results.

In [ ]:
!git clone -q https://github.com/hkmztrk/DeepDTA.git
print("Repository cloned.")

## 2b. pKd Conversion (Davis only) — Critical Fix

Davis' raw `Y` values are Kd in **nanomolar**, ranging up to 10,000+. Training directly on these raw values gives enormous MSE (in the millions) because the target scale itself is huge and unnormalized — this is *not* a sign the model is failing, it's a scaling issue.

The original paper converts Kd to **pKd** using a log transform before training:

```
pKd = -log10(Kd / 1e9)
```

This compresses the range to roughly 5–11, matching the scale the paper's reported metrics (MSE ≈ 0.261) are actually computed on.

**KIBA does not need this transform** — KIBA scores are already a unified composite score, not raw Kd/Ki/IC50 in nanomolar, so they're used as-is (matching the paper).

In [ ]:
def convert_to_pkd(Y):
    """Convert raw Kd (nanomolar) to pKd, matching the DeepDTA paper's preprocessing.
    Only applied to Davis -- KIBA scores are already on a usable scale."""
    Y = np.where(Y == 0, 1e-10, Y)  # avoid log(0)
    return -np.log10(Y / 1e9)

## 3. Character Sets

Neural networks need numeric input, not raw text. DeepDTA encodes each character in a SMILES string or protein sequence as an integer using a fixed lookup table (charset). These charsets are taken directly from the paper / original implementation.

- SMILES charset: 64 possible characters
- Protein charset: 25 possible characters (20 standard amino acids + a few rare/ambiguous codes)

In [ ]:
CHARISOSMISET = {"#": 1, "%": 2, ")": 3, "(": 4, "+": 5, "-": 6, "/": 7,
    ".": 8, "1": 9, "0": 10, "3": 11, "2": 12, "5": 13, "4": 14, "7": 15,
    "6": 16, "9": 17, "8": 18, "=": 19, "A": 20, "@": 21, "C": 22, "B": 23,
    "E": 24, "D": 25, "G": 26, "F": 27, "I": 28, "H": 29, "K": 30, "M": 31,
    "L": 32, "O": 33, "N": 34, "P": 35, "S": 36, "R": 37, "U": 38, "T": 39,
    "W": 40, "V": 41, "Y": 42, "[": 43, "Z": 44, "]": 45, "\\": 46, "a": 47,
    "c": 48, "b": 49, "e": 50, "d": 51, "g": 52, "f": 53, "i": 54, "h": 55,
    "m": 56, "l": 57, "o": 58, "n": 59, "s": 60, "r": 61, "u": 62, "t": 63, "y": 64}
CHARISOSMILEN = 64

CHARPROTSET = {"A": 1, "C": 2, "B": 3, "E": 4, "D": 5, "G": 6, "F": 7,
    "I": 8, "H": 9, "K": 10, "M": 11, "L": 12, "O": 13, "N": 14, "Q": 15,
    "P": 16, "S": 17, "R": 18, "U": 19, "T": 20, "W": 21, "V": 22, "Y": 23,
    "X": 24, "Z": 25}
CHARPROTLEN = 25

SMILES_MAXLEN = 100
PROTEIN_MAXLEN = 1000

print("SMILES charset size:", len(CHARISOSMISET))
print("Protein charset size:", len(CHARPROTSET))

## 4. Label Encoding Function

Converts a raw sequence (SMILES string or protein sequence) into a fixed-length integer array. Sequences shorter than `max_len` are zero-padded; longer sequences are truncated.

In [ ]:
def label_encode(sequence, charset, max_len):
    encoded = np.zeros(max_len, dtype=np.int64)
    for i, ch in enumerate(sequence[:max_len]):
        encoded[i] = charset.get(ch, 0)
    return encoded

## 5. Attention Pooling Layer (Custom Addition)

The original DeepDTA uses `GlobalMaxPooling1D` after the convolutional stack — it keeps only the single strongest activation per filter and discards positional information about *where* in the sequence that signal came from.

Here we add a lightweight **additive attention pooling layer**: instead of taking the max, the model learns a weight for every position in the sequence, normalizes these weights with softmax, and computes a weighted sum. This has two benefits:

1. It lets the model combine signal from multiple informative regions instead of just the single strongest one.
2. The learned attention weights are directly inspectable — we can visualize which characters in a SMILES string or which residues in a protein sequence the model focused on for a given prediction.

In [ ]:
class AttentionPooling1D(layers.Layer):
    """Learns a per-timestep importance score, normalizes it with softmax,
    and returns the weighted sum over the sequence dimension.
    Also exposes the attention weights for later visualization."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = layers.Dense(1)

    def call(self, inputs):
        # inputs shape: (batch, seq_len, channels)
        scores = self.score_dense(inputs)               # (batch, seq_len, 1)
        weights = tf.nn.softmax(scores, axis=1)          # (batch, seq_len, 1)
        weighted_sum = tf.reduce_sum(inputs * weights, axis=1)  # (batch, channels)
        return weighted_sum, weights

    def get_config(self):
        return super().get_config()

## 6. Model Architecture

Two parallel branches (drug and protein), each: Embedding → 3× Conv1D → Attention Pooling. The two pooled representations are concatenated and passed through 3 fully connected layers to a single regression output (predicted binding affinity).

Filter sizes (32 → 64 → 96) and dense layer sizes (1024 → 1024 → 512) match the paper's best-performing configuration. The protein branch uses a wider convolution window (kernel size 8) than the drug branch (kernel size 4), since protein sequences are much longer and need a wider receptive field to capture meaningful motifs.

In [ ]:
def build_deepdta_attention_model():
    # ---------- Drug (SMILES) branch ----------
    drug_input = layers.Input(shape=(SMILES_MAXLEN,), dtype='int64', name='drug_input')
    drug_embed = layers.Embedding(input_dim=CHARISOSMILEN + 1, output_dim=128,
                                   name='drug_embedding')(drug_input)
    x = layers.Conv1D(32, 4, activation='relu', padding='valid')(drug_embed)
    x = layers.Conv1D(64, 4, activation='relu', padding='valid')(x)
    x = layers.Conv1D(96, 4, activation='relu', padding='valid')(x)
    drug_pooled, drug_attn_weights = AttentionPooling1D(name='drug_attention')(x)

    # ---------- Protein branch ----------
    protein_input = layers.Input(shape=(PROTEIN_MAXLEN,), dtype='int64', name='protein_input')
    protein_embed = layers.Embedding(input_dim=CHARPROTLEN + 1, output_dim=128,
                                      name='protein_embedding')(protein_input)
    y = layers.Conv1D(32, 8, activation='relu', padding='valid')(protein_embed)
    y = layers.Conv1D(64, 8, activation='relu', padding='valid')(y)
    y = layers.Conv1D(96, 8, activation='relu', padding='valid')(y)
    protein_pooled, protein_attn_weights = AttentionPooling1D(name='protein_attention')(y)

    # ---------- Merge + fully connected head ----------
    merged = layers.concatenate([drug_pooled, protein_pooled])
    z = layers.Dense(1024, activation='relu')(merged)
    z = layers.Dropout(0.1)(z)
    z = layers.Dense(1024, activation='relu')(z)
    z = layers.Dropout(0.1)(z)
    z = layers.Dense(512, activation='relu')(z)
    output = layers.Dense(1, name='affinity_output')(z)

    prediction_model = Model(inputs=[drug_input, protein_input], outputs=output,
                              name='DeepDTA_Attention')

    # A second model that additionally exposes attention weights, for interpretability later
    attention_model = Model(inputs=[drug_input, protein_input],
                             outputs=[output, drug_attn_weights, protein_attn_weights],
                             name='DeepDTA_Attention_Interpretable')

    return prediction_model, attention_model

## 7. Reusable Evaluation Metrics

MSE is standard, but the paper additionally reports two metrics not available in scikit-learn:

- **Concordance Index (CI):** measures whether the model correctly ranks *pairs* of samples by relative affinity, not just absolute accuracy. 0.5 = random, 1.0 = perfect ranking.
- **rm²:** penalizes models that rank correctly but whose predicted values deviate from the ideal regression line — i.e. ranking is right but scale is off.

Both functions are defined once here and reused for Davis and KIBA.

In [ ]:
def concordance_index(y_true, y_pred):
    """Standard CI as used in the DeepDTA paper. O(n^2) — use a sample for large test sets."""
    pairs = 0
    correct = 0
    for i in range(len(y_true)):
        for j in range(len(y_true)):
            if y_true[i] > y_true[j]:
                pairs += 1
                if y_pred[i] > y_pred[j]:
                    correct += 1
                elif y_pred[i] == y_pred[j]:
                    correct += 0.5
    return correct / pairs if pairs > 0 else 0.0


def get_rm2(y_true, y_pred):
    from scipy import stats
    r2 = stats.pearsonr(y_true, y_pred)[0] ** 2
    slope, intercept, _, _, _ = stats.linregress(y_true, y_pred)
    y_pred_from_line = slope * y_true + intercept
    ss_res = np.sum((y_pred - y_pred_from_line) ** 2)
    ss_tot = np.sum((y_pred - np.mean(y_pred)) ** 2)
    r02 = 1 - (ss_res / ss_tot)
    return r2 * (1 - np.sqrt(abs(r2 - r02)))


def evaluate_predictions(y_true, y_pred, ci_sample_size=1000, label=""):
    mse = np.mean((y_true - y_pred) ** 2)
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(y_true), min(ci_sample_size, len(y_true)), replace=False)
    ci = concordance_index(y_true[sample_idx], y_pred[sample_idx])
    rm2 = get_rm2(y_true, y_pred)
    print(f"[{label}] MSE: {mse:.4f} | CI (sampled): {ci:.4f} | rm2: {rm2:.4f}")
    return {"mse": mse, "ci": ci, "rm2": rm2}

## 8. Full Pipeline Function

Wraps everything (load → encode → build pairs → split by official folds → train → evaluate) into one function, so we can call it once for Davis and once for KIBA without duplicating logic.

**Note on batch size:** the paper uses batch_size=256; this notebook defaults to a smaller **128**, which reduces memory pressure and adds a bit of extra gradient noise (mild regularization effect). Pass a different value to `run_deepdta_pipeline(..., batch_size=...)` if you want to experiment further (e.g. 32 or 64 for even smaller batches).

In [ ]:
def run_deepdta_pipeline(dataset_name, epochs=100, batch_size=128, quick_test=False):
    """dataset_name: 'davis' or 'kiba'"""
    path = f"DeepDTA/data/{dataset_name}/"

    # --- Load ---
    ligands = json.load(open(path + "ligands_can.txt"))
    proteins = json.load(open(path + "proteins.txt"))
    Y = np.array(pickle.load(open(path + "Y", "rb"), encoding='latin1'))

    # Critical fix: Davis needs raw Kd -> pKd log conversion before training.
    # KIBA scores are already on a usable scale and are used as-is.
    if dataset_name == "davis":
        Y = convert_to_pkd(Y)

    print(f"[{dataset_name}] drugs={len(ligands)} proteins={len(proteins)} Y_shape={Y.shape}")
    print(f"[{dataset_name}] Y range: {np.nanmin(Y):.3f} to {np.nanmax(Y):.3f}  (sanity check -- Davis should be ~5-11, not thousands)")

    # --- Encode ---
    drug_ids = list(ligands.keys())
    protein_ids = list(proteins.keys())
    encoded_drugs = np.array([label_encode(ligands[d], CHARISOSMISET, SMILES_MAXLEN) for d in drug_ids])
    encoded_proteins = np.array([label_encode(proteins[p], CHARPROTSET, PROTEIN_MAXLEN) for p in protein_ids])

    # --- Build (drug, protein, affinity) pairs, skipping missing values ---
    rows, cols = np.where(np.isfinite(Y))
    affinities = Y[rows, cols]
    print(f"[{dataset_name}] valid pairs: {len(affinities)} / {Y.size}")

    # --- Official train/test folds ---
    train_fold = json.load(open(path + "folds/train_fold_setting1.txt"))
    test_fold = json.load(open(path + "folds/test_fold_setting1.txt"))
    train_indices = list(itertools.chain(*train_fold))
    test_indices = test_fold

    X_train_drug = encoded_drugs[rows[train_indices]]
    X_train_protein = encoded_proteins[cols[train_indices]]
    y_train = affinities[train_indices]

    X_test_drug = encoded_drugs[rows[test_indices]]
    X_test_protein = encoded_proteins[cols[test_indices]]
    y_test = affinities[test_indices]

    print(f"[{dataset_name}] train samples: {len(y_train)} | test samples: {len(y_test)}")

    # --- Build + compile model ---
    model, attention_model = build_deepdta_attention_model()
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_squared_error'])

    if quick_test:
        epochs = 3

    # --- Train ---
    history = model.fit(
        x=[X_train_drug, X_train_protein],
        y=y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_data=([X_test_drug, X_test_protein], y_test),
        verbose=1
    )

    # --- Evaluate ---
    y_pred = model.predict([X_test_drug, X_test_protein]).flatten()
    metrics = evaluate_predictions(y_test, y_pred, label=dataset_name.upper())

    return {
        "model": model,
        "attention_model": attention_model,
        "history": history,
        "metrics": metrics,
        "X_test_drug": X_test_drug,
        "X_test_protein": X_test_protein,
        "y_test": y_test,
        "y_pred": y_pred,
        "ligands": ligands,
        "proteins": proteins,
    }

## 9. Run on Davis

Davis is smaller (68 drugs × 442 proteins), so we run it first to validate the full pipeline before committing to the larger KIBA run.

Set `quick_test=True` on first run to confirm everything works end-to-end in ~3 epochs before committing to a full 100-epoch training run.

In [ ]:
davis_results = run_deepdta_pipeline("davis", epochs=100, batch_size=128, quick_test=True)  # set quick_test=False for full run

## 10. Training Curves — Davis

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(davis_results["history"].history['loss'], label='Train Loss')
plt.plot(davis_results["history"].history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('DeepDTA + Attention — Training Curve (Davis)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11. Attention Visualization (Custom Addition)

This is the interpretability payoff of the attention layer: for a single test example, we can see exactly which characters in the SMILES string and which residues in the protein sequence the model weighted most heavily when making its prediction.

In [ ]:
def visualize_attention(results, sample_idx=0, dataset_label="davis"):
    model = results["attention_model"]
    drug_seq = results["X_test_drug"][sample_idx:sample_idx+1]
    protein_seq = results["X_test_protein"][sample_idx:sample_idx+1]

    pred, drug_attn, protein_attn = model.predict([drug_seq, protein_seq])
    drug_attn = drug_attn[0, :, 0]
    protein_attn = protein_attn[0, :, 0]

    fig, axes = plt.subplots(2, 1, figsize=(14, 6))

    conv_len_drug = len(drug_attn)  # shorter than SMILES_MAXLEN due to valid-padding conv
    axes[0].bar(range(conv_len_drug), drug_attn, color='steelblue')
    axes[0].set_title(f'Drug (SMILES) Attention Weights — {dataset_label} sample {sample_idx}')
    axes[0].set_xlabel('Position along convolved sequence')
    axes[0].set_ylabel('Attention weight')

    conv_len_protein = len(protein_attn)
    axes[1].bar(range(conv_len_protein), protein_attn, color='indianred')
    axes[1].set_title(f'Protein Attention Weights — {dataset_label} sample {sample_idx}')
    axes[1].set_xlabel('Position along convolved sequence')
    axes[1].set_ylabel('Attention weight')

    plt.tight_layout()
    plt.show()

    print(f"Predicted affinity: {pred[0][0]:.3f} | True affinity: {results['y_test'][sample_idx]:.3f}")

visualize_attention(davis_results, sample_idx=0, dataset_label="Davis")

## 12. Run on KIBA

Same pipeline, larger and sparser dataset (2,111 drugs × 229 proteins, with genuine missing values that `np.isfinite` filters out automatically — no code changes needed). Expect a longer training time; use `quick_test=True` first to validate.

In [ ]:
kiba_results = run_deepdta_pipeline("kiba", epochs=100, batch_size=128, quick_test=True)  # set quick_test=False for full run

## 13. Training Curves — KIBA

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(kiba_results["history"].history['loss'], label='Train Loss')
plt.plot(kiba_results["history"].history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('DeepDTA + Attention — Training Curve (KIBA)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 14. Attention Visualization — KIBA

In [ ]:
visualize_attention(kiba_results, sample_idx=0, dataset_label="KIBA")

## 15. Results Summary

Compare against the original paper's published results.

| Metric | Davis (this notebook) | Davis (paper) | KIBA (this notebook) | KIBA (paper) |
|---|---|---|---|---|
| MSE | *(fill in after full run)* | ~0.261 | *(fill in after full run)* | ~0.194 |
| CI | *(fill in after full run)* | ~0.878 | *(fill in after full run)* | ~0.863 |
| rm² | *(fill in after full run)* | ~0.630 | *(fill in after full run)* | ~0.673 |

Values close to the paper's (not necessarily identical, due to training randomness and the added attention mechanism) confirm a successful reproduction. Any consistent gap is worth discussing — e.g. whether attention pooling trades off some raw accuracy for interpretability, or whether it improves both.

In [ ]:
print("=== DAVIS ===")
print(davis_results["metrics"])
print()
print("=== KIBA ===")
print(kiba_results["metrics"])

## 16. Conclusion & Next Steps

This notebook reproduces DeepDTA's core two-branch CNN architecture for drug-target binding affinity prediction, and extends it with a custom attention pooling mechanism that adds interpretability without changing the fundamental architecture family.

**Possible extensions:**
- Compare attention pooling vs. the original max pooling directly (ablation study, same seeds)
- Cross-reference high-attention protein regions against known binding pocket residues (ties back to prior docking/structural work)
- Move to Step 2 of the roadmap: **GraphDTA**, which replaces the SMILES CNN branch with a graph neural network over the molecular graph directly

**Portfolio note:** this project reproduces a peer-reviewed paper's core method faithfully while adding a genuine, explainable extension (attention pooling + visualization) — not just re-running someone else's code.
